In [ ]:
JOBS_GPU0 = ['diagnostics']
JOBS_GPU1 = ['diagnostics']
JOBS_CPU = []
RUN_TAG = 'vna_v4_exact_diagnostics'
PRESET = 'EXACT_DIAGNOSTICS'
REPO_COMMIT = '8c56d02dab1ee16c47eb9138bf4aa3a64b9b45e2'
SOURCE_DIGEST = None
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000

MODE = 'profile'  # GPU0: D-exact + capacity; GPU1: reference then D-power10; M1 remains held


# T29–T31: diagnostics on up to two T4 GPUs
Attach CIFAR-100 and the old task-1 checkpoint or an output archive; checkpoint discovery scans all inputs automatically. Code loads directly from the pinned Git commit; no source ZIP is needed. GPU0 runs current D-exact with --capacity-profile immediately. GPU1 runs reference D (b97d25c, power10), then current D-power10; compare strictly at rel tol 1e-5. Power10 failure or mismatch is advisory and does not gate D-exact. With one GPU, run D-exact first, then both power10 checks serially. D-exact includes the same-state curvature/ranking audit and 100-class capacity proxy. Audit time is reported separately. M1 stays held and uses exact curvature.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna4_main")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() < 1:
        raise RuntimeError(f"Select at least one T4; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs')
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
OLD_REPO = REPO.with_name('counterfactual-projection-d88abc4')
OLD_COMMIT = 'd88abc496eb1fe21d1b6df66d90b47cf93d7ded1'
REFERENCE_REPO = REPO.with_name('counterfactual-projection-profile-reference')
REFERENCE_COMMIT = 'b97d25c9aaee4949acbb50869912042b5a47c1f3'
if MODE == 'profile':
    if not REFERENCE_REPO.exists(): private_clone(MAIN_URL, REFERENCE_REPO, REFERENCE_COMMIT)
    ref_head = subprocess.check_output(['git','-C',str(REFERENCE_REPO),'rev-parse','HEAD'],text=True).strip()
    ref_status = subprocess.check_output(['git','-C',str(REFERENCE_REPO),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if ref_head != REFERENCE_COMMIT or ref_status: raise RuntimeError('Reference D must be a clean b97d25c tree')
    for upstream in (REPO/'third_party').iterdir():
        destination = REFERENCE_REPO/'third_party'/upstream.name
        if upstream.is_dir() and not destination.exists(): destination.symlink_to(upstream, target_is_directory=True)
if OLD_REPO.exists():
    head = subprocess.run(['git','-C',str(OLD_REPO),'rev-parse','HEAD'],capture_output=True,text=True)
    status = subprocess.run(['git','-C',str(OLD_REPO),'status','--porcelain','--untracked-files=no'],capture_output=True,text=True)
    shared = (OLD_REPO/'.git/objects/info/alternates').exists()
    if shared or head.returncode or head.stdout.strip() != OLD_COMMIT or status.returncode or status.stdout.strip():
        print('Rebuilding incomplete or shared historical source checkout:', OLD_REPO)
        shutil.rmtree(OLD_REPO)
if not OLD_REPO.exists():
    private_clone(MAIN_URL, OLD_REPO, OLD_COMMIT)
head = subprocess.check_output(['git','-C',str(OLD_REPO),'rev-parse','HEAD'],text=True).strip()
status = subprocess.check_output(['git','-C',str(OLD_REPO),'status','--porcelain','--untracked-files=no'],text=True).strip()
if head != OLD_COMMIT or status: raise RuntimeError('Historical source checkout is not a clean d88abc4 tree')
for upstream in (REPO/'third_party').iterdir():
    destination = OLD_REPO/'third_party'/upstream.name
    if upstream.is_dir() and not destination.exists(): destination.symlink_to(upstream, target_is_directory=True)
script = REPO/'diagnostics/exact_solver.py'
from experiments.runners.pycil import upstream_imports
from diagnostics.exact_solver import find_task1_checkpoint
upstream_imports()
checkpoint = find_task1_checkpoint(Path('/kaggle/input'), STAGE_B/'exact_diagnostics'/'task1_source.pt')
from concurrent.futures import ThreadPoolExecutor, as_completed
diagnostic_root = STAGE_B/'exact_diagnostics'/MODE/REPO_COMMIT[:12]
branches = ('D-exact','R','D-power10') if MODE == 'profile' else ('A','B','C','D') if MODE == 'nan' else ()
if not branches: raise ValueError('MODE must be profile or nan')
gpu_count = min(2, torch.cuda.device_count())
if gpu_count < 1: raise RuntimeError('Diagnostics require at least one T4 GPU')
ENV['VNA_DIAGNOSTIC_LANES'] = str(gpu_count)
queues = [('D-exact',),('R','D-power10')] if MODE == 'profile' and gpu_count == 2 else [branches[gpu::gpu_count] for gpu in range(gpu_count)]
outcomes = {}
print('Diagnostic GPU queues:', {gpu: list(jobs) for gpu, jobs in enumerate(queues)}, flush=True)
def run_diagnostic_lane(gpu, jobs):
    for branch in jobs:
        tree = OLD_REPO if branch in ('B','C') else REFERENCE_REPO if branch=='R' else REPO
        directory = diagnostic_root/branch
        directory.mkdir(parents=True,exist_ok=True)
        command = [sys.executable,'-u',str(script),'--repo',str(tree),'--branch','D' if branch=='R' or branch.startswith('D-') else branch,
            '--checkpoint',str(checkpoint),'--data-dir',str(DATA_ROOT),'--out',str(directory),
            '--curvature','power10' if branch in ('R','D-power10','B','C') else 'exact']
        if MODE == 'profile': command.append('--profile')
        if MODE == 'profile' and branch == 'D-exact': command.append('--capacity-profile')
        try:
            if not launch(command,gpu,f'{MODE}_{branch}',RUN_ROOT/f'{MODE}_{branch}.log'):
                outcomes[branch] = {'status':'PAUSED'}
                return False
            outcomes[branch] = {'status':'COMPLETED'}
        except Exception as error:
            outcomes[branch] = {'status':'FAILED','error':str(error)}
            if MODE != 'profile' or branch == 'D-exact': raise
            print(f'Advisory power10 check {branch} failed: {error}',flush=True)
        archive_outputs()
    return True
try:
    with ThreadPoolExecutor(max_workers=gpu_count) as pool:
        pending = [pool.submit(run_diagnostic_lane, gpu, jobs) for gpu, jobs in enumerate(queues)]
        failures = []
        for future in as_completed(pending):
            try: future.result()
            except Exception as error: failures.append(error)
    if MODE == 'profile':
        from diagnostics.exact_solver import compare_profiles
        from experiments.io_utils import atomic_json_save
        def profile_result(branch):
            outcome = outcomes.get(branch,{'status':'NOT_RUN'})
            path = diagnostic_root/branch/'diagnostic.json'
            if outcome['status'] != 'COMPLETED': return outcome
            try: return json.loads(path.read_text())
            except (OSError,ValueError) as error: return {'status':'MISSING_REPORT','error':str(error)}
        reference, power, exact = (profile_result(branch) for branch in ('R','D-power10','D-exact'))
        stage1 = compare_profiles(reference,power,strict=True)
        stage1.update(advisory=True,execution=outcomes)
        atomic_json_save(stage1,diagnostic_root/'stage1_power10_vs_b97d25c.json')
        print(json.dumps(stage1,indent=2),flush=True)
        stage2 = compare_profiles(power,exact)
        stage2['same_state_curvature_and_rankings'] = exact.get('curvature_differences')
        stage2['exact_status'] = exact.get('status')
        atomic_json_save(stage2,diagnostic_root/'stage2_exact_vs_power10.json')
        print(json.dumps(stage2,indent=2),flush=True)
    if failures: raise failures[0]
finally:
    archive_outputs()
print('M1 remains held: review the actual profiles and A–D diagnostic reports.')


In [ ]:
for path in (STAGE_B/'exact_diagnostics').rglob('diagnostic.json'): print(path, path.read_text())
archive_outputs()
